# Notebook 16b — Offline v2 score validation (G1b)

Validates pre-registered v2 saliency variants against the 192 cached causal channel-ablation harms from Notebook 16. Fully offline: no GPU, no data loading, no re-ablation. Protocol frozen before computation; selection and confirmation use disjoint stratified halves (seed 0).

**Recorded outcome of the archived run:** chosen variant `v_c` (Fisher layer mass x within-layer ASVG leverage rank); G1b PASSED 4/4 harms vs Fisher on the holdout (AWBIR 0.782 vs 0.711), max deficit -0.045.

In [ ]:
# NB16b: v2 score variants validated OFFLINE against the 192 cached causal harms.
# Pre-registered protocol (frozen before computation):
#   - Variants: V-A denormalized SBL; V-B Fisher-form (squared-leverage) SBL;
#     V-C within-layer semantic rank x layer-level Fisher scale;
#     V-D per-channel Fisher x within-layer semantic rank.
#   - Profile aggregation for V-A/V-B: mean over the 3 cost profiles (frozen choice).
#   - Selection on a stratified random HALF of groups (seed 0, stratified by layer);
#     winner = most harms (of 4) beating Fisher's Spearman on that half,
#     tie-break = highest mean Spearman advantage.
#   - Confirmatory gate G1b on the OTHER half only, chosen variant only:
#     PASS = beats Fisher on >=2 of 4 harms AND trails Fisher by >0.10 on none.
#   - All other variants' holdout numbers are reported as exploratory.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

ROOT = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
R16  = ROOT / "results/saber/16_score_validation"
OUT  = ROOT / "results/saber/16b_score_v2"
OUT.mkdir(parents=True, exist_ok=True)

HARMS = ["harm_awbir", "harm_fine_macro_f1", "harm_family_macro_f1", "harm_hsr_balanced_soc"]

df = pd.read_csv(R16 / "score_and_causal_harm.csv")
assert len(df) == 192 and df["group_id"].is_unique

sbl = pd.concat([pd.read_csv(R16 / f"sbl_scores_{p}.csv")[["group_id", "sbl_raw"]]
                 for p in ["miss_sensitive", "balanced_soc", "alert_fatigue"]])
va = sbl.groupby("group_id")["sbl_raw"].mean().rename("v_a")

edges = pd.read_csv(R16 / "edge_group_leverage_long.csv")
vb = (edges.assign(sq=edges["edge_weight"] * edges["raw_boundary_leverage"] ** 2)
      .groupby(["group_id", "cost_profile"])["sq"].sum()
      .groupby("group_id").mean().rename("v_b"))

df = df.merge(va, on="group_id").merge(vb, on="group_id")
assert df[["v_a", "v_b"]].notna().all().all()

df["sem_rank"] = df.groupby("module_path")["v_a"].rank(pct=True)
layer_fisher = df.groupby("module_path")["fisher"].mean()
df["v_c"] = df["sem_rank"] * df["module_path"].map(layer_fisher)
df["v_d"] = df["sem_rank"] * df["fisher"]

VARIANTS = ["v_a", "v_b", "v_c", "v_d"]

rng = np.random.default_rng(0)
sel_mask = pd.Series(False, index=df.index)
for _, idx in df.groupby("module_path").groups.items():
    idx = np.array(list(idx))
    sel_mask.loc[rng.permutation(idx)[: len(idx) // 2]] = True
sel, hold = df[sel_mask], df[~sel_mask]
print(f"selection n={len(sel)}, holdout n={len(hold)}")

def table(frame, scores):
    rows = []
    for s in scores:
        for h in HARMS:
            rows.append({"score": s, "harm": h,
                         "spearman": spearmanr(frame[s], frame[h]).correlation})
    return pd.DataFrame(rows)

sel_tab = table(sel, VARIANTS + ["fisher"]).pivot(index="score", columns="harm", values="spearman")
fisher_sel = sel_tab.loc["fisher"]
wins = (sel_tab.loc[VARIANTS] > fisher_sel).sum(axis=1)
adv  = (sel_tab.loc[VARIANTS] - fisher_sel).mean(axis=1)
chosen = wins.sort_values(ascending=False).index[
    np.argmax(adv.loc[wins == wins.max()].values)] if (wins == wins.max()).sum() > 1 else wins.idxmax()
print("\n== selection half ==\n", sel_tab.round(3).to_string())
print(f"\nwins vs fisher: {wins.to_dict()} | mean advantage: {adv.round(3).to_dict()}")
print(f"CHOSEN: {chosen}")

hold_tab = table(hold, VARIANTS + ["fisher", "taylor", "magnitude", "r_sbl"]) \
           .pivot(index="score", columns="harm", values="spearman")
fisher_hold, chosen_hold = hold_tab.loc["fisher"], hold_tab.loc[chosen]
n_won   = int((chosen_hold > fisher_hold).sum())
max_def = float((fisher_hold - chosen_hold).max())
passed  = bool(n_won >= 2 and max_def <= 0.10)
print("\n== holdout half ==\n", hold_tab.round(3).to_string())
print(f"\nG1b: chosen={chosen} | harms won={n_won}/4 | max deficit={max_def:.3f} | PASSED={passed}")

df[["group_id", "module_path", "channel_index"] + VARIANTS + ["sem_rank"]] \
    .to_csv(OUT / "v2_scores.csv", index=False)
sel_tab.to_csv(OUT / "v2_selection_report.csv")
hold_tab.to_csv(OUT / "v2_holdout_report.csv")
gate = {"gate": "G1b_score_v2_offline", "passed": passed, "chosen_variant": chosen,
        "protocol": "stratified half-split seed0; select-most-wins-vs-fisher; "
                    "confirm on holdout; pass = >=2/4 harms won and max deficit <=0.10",
        "harms_won": n_won, "max_deficit_vs_fisher": max_def,
        "holdout": {h: {"chosen": float(chosen_hold[h]), "fisher": float(fisher_hold[h])}
                    for h in HARMS}}
(OUT / "G1b_score_gate.json").write_text(json.dumps(gate, indent=2))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, h in zip(axes, HARMS):
    ax.scatter(hold[chosen], hold[h], alpha=0.6, s=18)
    ax.set_xlabel(chosen); ax.set_ylabel(h)
    ax.set_title(f"{h.replace('harm_','')}: rho={spearmanr(hold[chosen], hold[h]).correlation:.2f}")
plt.tight_layout(); plt.savefig(OUT / "v2_holdout_scatter.png", dpi=120); plt.show()
print("\nSaved to", OUT)
